# Usage in Python

You can use Imaging Server Kit in Python to interact with local or remote algorithms, run computations tile-by-tile, load samples, access documentation, and more.

## Running algorithms

Let's consider our usual thresholding algorithm:

In [ ]:
import imaging_server_kit as sk
import skimage.data

@sk.algorithm(
    name="Intensity threshold",
    parameters={"threshold": sk.Integer(name="Threshold", min=0, max=255, default=128)},
    samples=[{"image" : skimage.data.coins()}],
)
def threshold_algo(image, threshold):
    mask = image > threshold
    return sk.Mask(mask, name="Binary mask")

Our function turned algorithm still behaves like the original Python function. If you try this:

In [ ]:
image = skimage.data.coins()

mask = threshold_algo(image, threshold=100)
print(mask)

You will get a standard Numpy array for `mask`.

However, one notable difference is that **parameters are validated** at runtime according to the annotations you provided.

Because of that, and your annotation of `min=0` for the `threshold` parameter, you won't be able to run the algorithm with negative threshold values:

In [ ]:
mask_invalid = threshold_algo(image, threshold=-1)  # Will raise a `ValidationError`

In this case, a `ValidationError` is raised with the message indicating that the threshold input should be greater than or equal to zero.

Moreover, the algorithm object has gained a few extra methods. You can execute computations via `.run()`:

In [ ]:
results = threshold_algo.run(image, threshold=100)

print(results)

In this case, the returned `results` is an instance of a `Stack` object from `imaging-server-kit`. `Stack` encapsulate a **stack of data layers**. In this case, the results include one layer containing the segmentation mask.

Data layers in a `Stack` object can be accessed either by name via `.read()` or by indexing (`results[0]`).

In [ ]:
mask_result = results.read("Binary mask")  # <- Equivalent to accessing `results[0]`

print(mask_result)

Here, `mask_result` is a `Mask` object, which is an instance of `sk.Layer`. You can retrieve the segmentation mask as its `data` attribute.

In [ ]:
mask = mask_result.data  # Numpy array

print(mask.shape)

All data layers have a `data` attribute and a few other attributes and methods you can explore:

In [ ]:
import inspect

print(inspect.getdoc(sk.Layer))

The `Stack` class has a few methods and attributes of its own:  

In [ ]:
print(inspect.getdoc(sk.Stack))

### Display a stack in Napari

You can use `sk.convert(stack, to="napari")` to create a Napari viewer and display a stack in it.

### More on `run`

A few more options when using `run` in Python:

- **`run(..., stack=...)`** accumulates results into an existing Stack.
- **`run(..., stack=viewer)`** sends results straight into a Napari viewer.
- `sk.run(algo, ...)` is equivalent to `algo.run(...)`

## Using `sk.Client`

````{note}
You need to run the "server-side" code from this section in a Python script (not a Jupyter notebook) as explained in [Serving algorithms](./07_server) via:

```python
...

if __name__ == "__main__":
  sk.serve(threshold_algo)
```
````

One of the main advantages of using `.run()` is that it will work in the same way when applied to a local algorithm and when applied to a **client instance connected to an algorithm server**.

To demonstrate this, let's serve our threshold algorithm so that it becomes available on http://localhost:8000. Once the server is available, you can connect to it from Python via a `sk.Client()` instance:

In [ ]:
import imaging_server_kit as sk

# Connect to the server
client = sk.Client("http://localhost:8000")

# Send the image and parameters, computation runs on the server, results are retrieved:
results = client.run(image, threshold=50)
print(results)

# Mask data layer
mask_result = results[0]
print(mask_result)

# Segmentation mask
mask = results[0].data
print(mask.shape)

Moreover, you can retrieve samples using the `.get_sample()` method on an algorithm, or client, and passing the index of the sample:

In [ ]:
sample = client.get_sample(idx=0)  # <- Retreive the first sample.

sample

Finally, using `.info()` on an algorithm, or a connected client, will open the algo docs page in a web browser:

In [ ]:
client.info()

## Working with algorithm collections

Algorithm collections expose the same methods as standalone algorithms, except that you need to provide a value for the `algorithm` parameter, which corresponds to the name of the algorithm.

On the **server side**, combine two algorithms and serve them:

In [ ]:
import imaging_server_kit as sk
import skimage.data
from skimage.filters import threshold_otsu, threshold_li

@sk.algorithm(
    name="intensity-threshold",
    parameters={"threshold": sk.Integer(name="Threshold", min=0, max=255, default=128)},
    samples=[{"image" : skimage.data.coins()}],
    tileable=True,
)
def threshold_algo(image, threshold):
    mask = image > threshold
    return sk.Mask(mask, name="Binary mask")


@sk.algorithm(
  name="automatic-threshold",
  parameters={"method": sk.Choice(name="Method", items=["otsu", "li"], default="otsu")},
  samples=[{"image" : skimage.data.coins()}],
  tileable=True,
)
def auto_threshold(image, method):
  if method == "otsu":
    mask = image > threshold_otsu(image)
  elif method == "li":
    mask = image > threshold_li(image)
  return sk.Mask(mask, name="Binary mask")

# Combine the algorithms
threshold_algos = sk.combine([threshold_algo, auto_threshold], name="threshold-algos")

if __name__ == "__main__":
  sk.serve(threshold_algos)

Then, on the **client side**:

In [ ]:
client = sk.Client("http://localhost:8000")

# List the available algorithms
print(client.algorithms)  # ["intensity-threshold", "automatic-threshold"]

thresh_results = client.run(algorithm="intensity-threshold", image=image, threshold=30)
print(thresh_results)

auto_results = client.run(algorithm="automatic-threshold", image=image, method="otsu")
print(auto_results)

## Tiled inference

To run an algorithm tile-by-tile, you can set `tiled=True` and specify the `tile_size`, `tile_overlap`, `tile_randomize` and `tile_delay` parameters:

In [ ]:
results = client.run(
  image=image,
  tiled=True,  # <- Enable tiled inference
  tile_size=64,  #  (64, 64) sized tiles. Could also be, e.g., (32, 64) for anisotripic nD tiles.
  tile_overlap=0.1,  # 10% overlap
  tile_randomize=True,  # Process the tiles in a random order
  tile_delay=0.0,  # (Optional) Add a little time delay betwen each tile
)

This will only work with algorithms that have enabled tiling with `tileable=True`.

Tiling works both with remote runs via a client, and with local runs (`algo.run(image, tiled=True, ...)`).

Tile results are assembled differently depending on the layers:

- Images are averaged in overlapping regions.
- With Masks, the last tile overrides overlapping regions by default. Instance segmentation needs sk.Mask(..., merger="instances"): labels are made unique across tiles, and objects crossing tile borders are stitched (this requires overlap).
- With Points, Vectors, and Boxes, each object goes to the tile containing it (with Vectors based on their origin, and with Boxes based on their center), with coordinates local to the tile. When overlap is used, objects in overlapping regions are returned once per tile.

## Regions and positions

Every layer has a `position` property that sets where it sits in **global pixel coordinates**. The layer's data is expressed in *local* coordinates, and `position` is interpreted as an offset that translates them to global coordinates.

You can set the position when creating a layer:

In [ ]:
import numpy as np

# With an image:
img = sk.Image(np.zeros((100, 200)), position=(50, 20))

print(img)             # <Image 'Image' float64 (100, 200), at (50, 20)>
print(img.position)    # (50, 20)
print(img.extent)      # Domain(position=(50, 20), size=(100, 200))
print(img.coords_min)  # (50.0, 20.0)
print(img.coords_max)  # (150.0, 220.0)

# With points:
pts = sk.Points(np.array([[30, 10], [60, 90]]))

print(pts.position)    # (0, 0)
print(pts.coords_min)  # (30.0, 10.0)
print(pts.coords_max)  # (61.0, 91.0)

# With a stack:
stack = sk.Stack([img, pts])

print(stack.extent)   # Domain(position=(30, 10), size=(120, 210))

stack.position = (10, 10)

print(img.position)   # (60, 30)
print(pts.position)   # (10, 10)

Layers and stacks both have the following properties:

| Property | Type | Description |
|---|---|---|
| `position` | `tuple` | Offset of the layer's local coordinates in global pixel space. |
| `extent` | `sk.Domain` | Region covered by the data in global coordinates. For images/masks, it starts at `position` and has the shape of the array. For objects (`Points`, `Vectors`...) it is the **bounding box of the objects**. For a `Stack`, it is the smallest region that contains all its layers. |
| `coords_min` | `tuple` | Lower corner of the extent (top-left in 2D) |
| `coords_max` | `tuple` | Upper corner of the extent (bottom-right in 2D) |
| `size` | `tuple` | Size of the extent along each axis |
| `ndim` | `int` | Number of spatial dimensions |

Extents are `sk.Domain` objects. A domain is a box defined by a `size` and a `position` (its lower corner, which defaults to the origin). You can create domains yourself to describe a region of interest:

In [ ]:
roi = sk.Domain(position=(20, 30), size=(60, 80))

print(roi.coords_min)  # (20.0, 30.0)
print(roi.coords_max)  # (80.0, 110.0)

### Restricting computation to a domain

You can limit a computation to a region with `algo.run(..., domain=roi)`. The returned stack will inherit the position and size of the provided domain.

Example:

```python
import imaging_server_kit as sk
from imaging_server_kit.demo.tools import otsu_threshold
from skimage.data import coins

# Run Otsu's threshold on a sub-region of the image
mask = otsu_threshold.run(image=coins(), domain=sk.Domain(position=(20, 30), size=(60, 80)))[0]

print(mask)  # <Mask 'Binary mask (auto)' bool (60, 80), at (20, 30)>
```

### Selecting data in layers and stacks

Domains can also be used to extract part of a layer or stack with `.select(domain=roi)`. For example:

In [ ]:
roi = sk.Domain(position=(20, 30), size=(60, 80))

image = sk.Image(data=coins())

image_roi = image.select(roi)

print(image_roi)  # <Image (60, 80), at (20, 30)>

You can also use Numpy-like indexing to select data in layers, or stacks:

In [ ]:
img = sk.Image(np.zeros((200, 200)))
pts = sk.Points(np.array([[30, 10], [60, 90], [120, 110]]))

print(img[50:150])  # <Image (100, 200), at (50, 0)>
print(pts[50:150])  # <Points 2 points, at (50, 0)>

stack = sk.Stack([img, pts])

roi = stack[:, 50:150]  # <- When indexing on stacks, the first "axis" is the layers

print(roi)  # [<Image (100, 200), at (50, 0)>, <Points 2 points, at (50, 0)>]

### Merging layers

You can use `merge_layers()` to merge a list of layers of the same kind into one:

In [ ]:
img1 = ...  # TODO: complete this
img2 = ...

img3 = sk.merge_layers([img1, img2])

print(img3)

## Summary
- Use `sk.Client()` to connect to algorithm servers from Python.
- Use `.run()` on a client, or the original algorithm, to retrieve a `Stack` object.
- `Stack` contain a stack of **data layers**.
- You can access the `data`, `name` and `meta` attributes of data layers.
- Parameters are validated (invalid parameters raise a `ValidationError`).
- Use `.get_sample()` to retrieve samples
- Use `.info()` to access documentation.
- Use `tiled=True` for tiled inference.
- Layers and stacks have a `position` and an `extent` in global pixel coordinates.

## Next steps

Next, we'll take a look at the different data layers available.